In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("hotel_bookings_clean.csv")

print("Shape:", df.shape)
df.head()

Shape: (72266, 37)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,total_guests,total_nights,family,arrival_month_num,season
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,0.0,0.0,0.0,Check-Out,2015-07-01,2.0,0.0,0.0,7.0,Summer
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,0.0,0.0,0.0,Check-Out,2015-07-01,2.0,0.0,0.0,7.0,Summer
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,75.0,0.0,0.0,Check-Out,2015-07-02,1.0,1.0,0.0,7.0,Summer
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,75.0,0.0,0.0,Check-Out,2015-07-02,1.0,1.0,0.0,7.0,Summer
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,98.0,0.0,1.0,Check-Out,2015-07-03,2.0,2.0,0.0,7.0,Summer


In [4]:
print(df["is_canceled"].value_counts())
print(df["is_canceled"].value_counts(normalize=True) * 100)

is_canceled
0    48257
1    24009
Name: count, dtype: int64
is_canceled
0    66.776908
1    33.223092
Name: proportion, dtype: float64


In [5]:
X = df.drop(columns=["is_canceled"])
y = df["is_canceled"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (72266, 36)
y shape: (72266,)


In [6]:
columns_to_drop = [
    "reservation_status",
    "reservation_status_date",
    "agent",
    "company",
    "estimated_revenue"
]

X = X.drop(columns=columns_to_drop, errors="ignore")

print("Remaining columns:")
print(X.columns.tolist())

Remaining columns:
['hotel', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'total_guests', 'total_nights', 'family', 'arrival_month_num', 'season']


In [7]:
print("Number of features:", X.shape[1])
print(X.dtypes)

Number of features: 32
hotel                              object
lead_time                           int64
arrival_date_year                   int64
arrival_date_month                 object
arrival_date_week_number            int64
arrival_date_day_of_month           int64
stays_in_weekend_nights             int64
stays_in_week_nights                int64
adults                              int64
children                          float64
babies                              int64
meal                               object
country                            object
market_segment                     object
distribution_channel               object
is_repeated_guest                   int64
previous_cancellations              int64
previous_bookings_not_canceled      int64
reserved_room_type                 object
assigned_room_type                 object
booking_changes                     int64
deposit_type                       object
days_in_waiting_list              float64
customer_ty

In [8]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numeric features:
['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'total_guests', 'total_nights', 'family', 'arrival_month_num']

Categorical features:
['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type', 'season']


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (57812, 32)
Testing set: (14454, 32)


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    )
}

In [12]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

results = []

for name, model in models.items():

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC_AUC": roc_auc_score(y_test, y_prob)
    })

baseline_results = pd.DataFrame(results)

baseline_results.sort_values(
    by="ROC_AUC",
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
2,Random Forest,0.883700,0.873593,0.759892,0.812785,0.944893
3,Gradient Boosting,0.864536,0.843644,0.726989,0.780984,0.933681
0,Logistic Regression,0.827245,0.772266,0.680758,0.723630,0.893969
1,Decision Tree,0.840667,0.756098,0.768222,0.762111,0.823789


In [13]:
from sklearn.model_selection import RandomizedSearchCV

In [14]:
gb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", GradientBoostingClassifier(random_state=42))
    ]
)

In [15]:
gb_param_grid = {
    "model__n_estimators": [100, 150, 200, 250],
    "model__learning_rate": [0.03, 0.05, 0.1, 0.15],
    "model__max_depth": [2, 3, 4, 5],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__subsample": [0.8, 0.9, 1.0]
}

In [16]:
gb_random = RandomizedSearchCV(
    estimator=gb_pipeline,
    param_distributions=gb_param_grid,
    n_iter=20,
    scoring="roc_auc",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

gb_random.fit(X_train, y_train)

Fitting 3 folds for each of 20 candidates, totalling 60 fits


RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(transformers=[('num',
                                                                               Pipeline(steps=[('imputer',
                                                                                                SimpleImputer(strategy='median')),
                                                                                               ('scaler',
                                                                                                StandardScaler())]),
                                                                               ['lead_time',
                                                                                'arrival_date_year',
                                                                                'arrival_date_week_number',
                                                                                'arrival_date_day_of_month',
                                                                                'stays_in_weekend_nights',
                                                                                'stays_in_week_nights',
                                                                                'adults',
                                                                                'childre...
                                             ('model',
                                              GradientBoostingClassifier(random_state=42))]),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'model__learning_rate': [0.03, 0.05,
                                                                 0.1, 0.15],
                                        'model__max_depth': [2, 3, 4, 5],
                                        'model__min_samples_leaf': [1, 2, 4],
                                        'model__min_samples_split': [2, 5, 10],
                                        'model__n_estimators': [100, 150, 200,
                                                                250],
                                        'model__subsample': [0.8, 0.9, 1.0]},
                   random_state=42, scoring='roc_auc', verbose=1)

In [17]:
print("Best Gradient Boosting parameters:")
print(gb_random.best_params_)

print("\nBest CV ROC-AUC:")
print(gb_random.best_score_)

Best Gradient Boosting parameters:
{'model__subsample': 0.8, 'model__n_estimators': 150, 'model__min_samples_split': 2, 'model__min_samples_leaf': 4, 'model__max_depth': 5, 'model__learning_rate': 0.1}

Best CV ROC-AUC:
0.9429202090222987


In [18]:
gb_best = gb_random.best_estimator_

gb_pred = gb_best.predict(X_test)
gb_prob = gb_best.predict_proba(X_test)[:, 1]

gb_tuned_results = {
    "Model": "Tuned Gradient Boosting",
    "Accuracy": accuracy_score(y_test, gb_pred),
    "Precision": precision_score(y_test, gb_pred),
    "Recall": recall_score(y_test, gb_pred),
    "F1": f1_score(y_test, gb_pred),
    "ROC_AUC": roc_auc_score(y_test, gb_prob)
}

gb_tuned_results

{'Model': 'Tuned Gradient Boosting',
 'Accuracy': 0.8787186937871869,
 'Precision': 0.8460839954597049,
 'Recall': 0.7761349437734277,
 'F1': 0.8096013902465515,
 'ROC_AUC': np.float64(0.9459429828157317)}

In [19]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [20]:
rf_param_grid = {
    "model__n_estimators": [200, 300, 400],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2"]
}

In [21]:
rf_random = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_param_grid,
    n_iter=20,
    scoring="roc_auc",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_random.fit(X_train, y_train)

Fitting 3 folds for each of 20 candidates, totalling 60 fits


/usr/local/lib/python3.13/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(transformers=[('num',
                                                                               Pipeline(steps=[('imputer',
                                                                                                SimpleImputer(strategy='median')),
                                                                                               ('scaler',
                                                                                                StandardScaler())]),
                                                                               ['lead_time',
                                                                                'arrival_date_year',
                                                                                'arrival_date_week_number',
                                                                                'arrival_date_day_of_month',
                                                                                'stays_in_weekend_nights',
                                                                                'stays_in_week_nights',
                                                                                'adults',
                                                                                'childre...
                                                                                'customer_type',
                                                                                'season'])])),
                                             ('model',
                                              RandomForestClassifier(n_jobs=-1,
                                                                     random_state=42))]),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'model__max_depth': [None, 10, 20, 30],
                                        'model__max_features': ['sqrt', 'log2'],
                                        'model__min_samples_leaf': [1, 2, 4],
                                        'model__min_samples_split': [2, 5, 10],
                                        'model__n_estimators': [200, 300, 400]},
                   random_state=42, scoring='roc_auc', verbose=1)

In [22]:
print("Best Random Forest parameters:")
print(rf_random.best_params_)

print("\nBest CV ROC-AUC:")
print(rf_random.best_score_)

Best Random Forest parameters:
{'model__n_estimators': 400, 'model__min_samples_split': 5, 'model__min_samples_leaf': 2, 'model__max_features': 'sqrt', 'model__max_depth': 30}

Best CV ROC-AUC:
0.9392806687668966


In [23]:
rf_best = rf_random.best_estimator_

rf_pred = rf_best.predict(X_test)
rf_prob = rf_best.predict_proba(X_test)[:, 1]

rf_tuned_results = {
    "Model": "Tuned Random Forest",
    "Accuracy": accuracy_score(y_test, rf_pred),
    "Precision": precision_score(y_test, rf_pred),
    "Recall": recall_score(y_test, rf_pred),
    "F1": f1_score(y_test, rf_pred),
    "ROC_AUC": roc_auc_score(y_test, rf_prob)
}

rf_tuned_results

{'Model': 'Tuned Random Forest',
 'Accuracy': 0.8796872837968729,
 'Precision': 0.8938030341990229,
 'Recall': 0.7238650562265723,
 'F1': 0.7999079507536532,
 'ROC_AUC': np.float64(0.9461218435715328)}

In [24]:
preprocessor_fitted = gb_best.named_steps["preprocessor"]

feature_names = preprocessor_fitted.get_feature_names_out()

model = gb_best.named_steps["model"]

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

feature_importance.head(20)

,Feature,Importance
0,num__lead_time,0.138600
1,num__arrival_date_year,0.134424
2,cat__market_segment_Online TA,0.093688
3,num__total_of_special_requests,0.088190
4,cat__country_PRT,0.086443
5,cat__hotel_City Hotel,0.084382
6,num__required_car_parking_spaces,0.069537
7,cat__hotel_Resort Hotel,0.069461
8,num__previous_cancellations,0.027377
9,cat__deposit_type_Non Refund,0.027036


In [25]:
baseline_results.to_csv(
    "baseline_model_results.csv",
    index=False
)

pd.DataFrame([
    gb_tuned_results,
    rf_tuned_results
]).to_csv(
    "tuned_model_results.csv",
    index=False
)

feature_importance.to_csv(
    "feature_importance.csv",
    index=False
)

In [26]:
import joblib

joblib.dump(
    gb_best,
    "hotel_cancellation_model.pkl"
)

['hotel_cancellation_model.pkl']

In [27]:
feature_importance.head(20)

,Feature,Importance
0,num__lead_time,0.138600
1,num__arrival_date_year,0.134424
2,cat__market_segment_Online TA,0.093688
3,num__total_of_special_requests,0.088190
4,cat__country_PRT,0.086443
5,cat__hotel_City Hotel,0.084382
6,num__required_car_parking_spaces,0.069537
7,cat__hotel_Resort Hotel,0.069461
8,num__previous_cancellations,0.027377
9,cat__deposit_type_Non Refund,0.027036


In [28]:
gb_tuned_results


{'Model': 'Tuned Gradient Boosting',
 'Accuracy': 0.8787186937871869,
 'Precision': 0.8460839954597049,
 'Recall': 0.7761349437734277,
 'F1': 0.8096013902465515,
 'ROC_AUC': np.float64(0.9459429828157317)}

In [29]:
rf_tuned_results

{'Model': 'Tuned Random Forest',
 'Accuracy': 0.8796872837968729,
 'Precision': 0.8938030341990229,
 'Recall': 0.7238650562265723,
 'F1': 0.7999079507536532,
 'ROC_AUC': np.float64(0.9461218435715328)}

In [30]:
import joblib

joblib.dump(
    gb_best,
    "hotel_cancellation_model.pkl"
)

print("Final model saved successfully.")

Final model saved successfully.


In [31]:
import joblib

joblib.dump(
    gb_best,
    "hotel_cancellation_model.pkl"
)

print("Final model saved successfully.")

Final model saved successfully.


In [32]:
print("Dataset shape:", df.shape)

print("\nFinal model:")
print(type(gb_best))

print("\nTuned Gradient Boosting:")
print(gb_tuned_results)

print("\nTuned Random Forest:")
print(rf_tuned_results)

print("\nTop 10 features:")
display(feature_importance.head(10))

Dataset shape: (72266, 37)

Final model:
<class 'sklearn.pipeline.Pipeline'>

Tuned Gradient Boosting:
{'Model': 'Tuned Gradient Boosting', 'Accuracy': 0.8787186937871869, 'Precision': 0.8460839954597049, 'Recall': 0.7761349437734277, 'F1': 0.8096013902465515, 'ROC_AUC': np.float64(0.9459429828157317)}

Tuned Random Forest:
{'Model': 'Tuned Random Forest', 'Accuracy': 0.8796872837968729, 'Precision': 0.8938030341990229, 'Recall': 0.7238650562265723, 'F1': 0.7999079507536532, 'ROC_AUC': np.float64(0.9461218435715328)}

Top 10 features:


,Feature,Importance
0,num__lead_time,0.138600
1,num__arrival_date_year,0.134424
2,cat__market_segment_Online TA,0.093688
3,num__total_of_special_requests,0.088190
4,cat__country_PRT,0.086443
5,cat__hotel_City Hotel,0.084382
6,num__required_car_parking_spaces,0.069537
7,cat__hotel_Resort Hotel,0.069461
8,num__previous_cancellations,0.027377
9,cat__deposit_type_Non Refund,0.027036


In [33]:
# Combine baseline and tuned model results

tuned_results = pd.DataFrame([
    gb_tuned_results,
    rf_tuned_results
])

final_model_comparison = pd.concat(
    [baseline_results, tuned_results],
    ignore_index=True
)

final_model_comparison

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.827245,0.772266,0.680758,0.723630,0.893969
1,Decision Tree,0.840667,0.756098,0.768222,0.762111,0.823789
2,Random Forest,0.883700,0.873593,0.759892,0.812785,0.944893
3,Gradient Boosting,0.864536,0.843644,0.726989,0.780984,0.933681
4,Tuned Gradient Boosting,0.878719,0.846084,0.776135,0.809601,0.945943
5,Tuned Random Forest,0.879687,0.893803,0.723865,0.799908,0.946122


In [34]:
baseline_results

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.827245,0.772266,0.680758,0.723630,0.893969
1,Decision Tree,0.840667,0.756098,0.768222,0.762111,0.823789
2,Random Forest,0.883700,0.873593,0.759892,0.812785,0.944893
3,Gradient Boosting,0.864536,0.843644,0.726989,0.780984,0.933681


In [35]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

rf_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['lead_time',
                                                   'arrival_date_year',
                                                   'arrival_date_week_number',
                                                   'arrival_date_day_of_month',
                                                   'stays_in_weekend_nights',
                                                   'stays_in_week_nights',
                                                   'adults', 'children',
                                                   'babies',
                                                   'is_repeated_guest',
                                                   'p...
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['hotel',
                                                   'arrival_date_month', 'meal',
                                                   'country', 'market_segment',
                                                   'distribution_channel',
                                                   'reserved_room_type',
                                                   'assigned_room_type',
                                                   'deposit_type',
                                                   'customer_type',
                                                   'season'])])),
                ('model',
                 RandomForestClassifier(n_estimators=200, n_jobs=-1,
                                        random_state=42))])

In [36]:
rf_model = rf_pipeline.named_steps["model"]
rf_preprocessor = rf_pipeline.named_steps["preprocessor"]

rf_feature_names = rf_preprocessor.get_feature_names_out()

rf_feature_importance = pd.DataFrame({
    "Feature": rf_feature_names,
    "Importance": rf_model.feature_importances_
})

rf_feature_importance = rf_feature_importance.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

In [37]:
rf_feature_importance.head(20)

,Feature,Importance
0,num__lead_time,0.104912
1,num__arrival_date_year,0.084140
2,num__adr,0.075030
3,num__total_of_special_requests,0.048888
4,num__arrival_date_day_of_month,0.048737
5,num__arrival_date_week_number,0.044418
6,cat__country_PRT,0.042681
7,num__total_nights,0.034326
8,num__stays_in_week_nights,0.031254
9,num__required_car_parking_spaces,0.031120


In [38]:
rf_feature_importance.to_csv(
    "rf_feature_importance.csv",
    index=False
)

print("Random Forest feature importance saved successfully.")

Random Forest feature importance saved successfully.


In [42]:
joblib.dump(
    rf_pipeline,
    "hotel_cancellation_model.pkl"
)

['hotel_cancellation_model.pkl']